# Customer Data Cleansing Pipeline

## Overview
This notebook reads customer data from CSV, cleanses it, and writes to a Delta table.

## Parameters (Widgets)
This notebook uses widgets for configuration:
* **input_csv_path**: Path to input CSV file (default: `/Volumes/csvfiles/default/demovol/customers_csv.csv`)
* **output_table_name**: Unity Catalog table name for output (default: `csvfiles.default.cleaned_customers`)
* **fill_missing_email**: Whether to fill missing emails (default: `yes`)

## Transformations Applied
1. **Handle Missing Values**
   - Fill missing numeric values (Age) with median
   - Fill missing text values (City, Customer_Name, Email) with appropriate defaults
   - Drop rows where ID is missing (critical field)

2. **Standardize Formats**
   - Trim whitespace from all string columns
   - Standardize City names to title case
   - Remove duplicate records based on ID

3. **Data Quality**
   - Validate Age is positive
   - Ensure ID is unique

---

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Create widgets for parameters
dbutils.widgets.text("input_csv_path", "/Volumes/csvfiles/default/demovol/customers_csv.csv", "Input CSV Path")
dbutils.widgets.text("output_table_name", "csvfiles.default.cleaned_customers", "Output Delta Table")
dbutils.widgets.dropdown("fill_missing_email", "yes", ["yes", "no"], "Fill Missing Emails?")

# Get parameter values
csv_path = dbutils.widgets.get("input_csv_path")
output_table = dbutils.widgets.get("output_table_name")
fill_email = dbutils.widgets.get("fill_missing_email")

print(f"Reading CSV file from: {csv_path}")

# Read CSV file
raw_df = spark.read.csv(
    csv_path,
    header=True,
    inferSchema=True
)

print(f"\n Successfully read {raw_df.count()} records")
print("\n Schema:")
raw_df.printSchema()

print("\n Sample data (first 5 rows):")
display(raw_df.limit(5))

Reading CSV file from: /Volumes/csvfiles/default/demovol/customers_csv.csv

 Successfully read 20 records

 Schema:
root
 |-- ID: integer (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Email: string (nullable = true)


 Sample data (first 5 rows):


ID,Customer_Name,City,Age,Email
1,Raj,Delhi,21,raj@gmail.com
2,Amit,Mumbai,35,amit@gmail.com
3,null,Delhi,42,priya@gmail.com
4,Neha,pune,29,null
5,Rohit,Bangalore,null,rohit@gmail.com


In [0]:
print("Checking data quality issues...\n")

# Check for missing values
print("\n Missing Values Count:")
missing_counts = raw_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in raw_df.columns]
)
display(missing_counts)

# Check for duplicates based on ID
duplicates_count = raw_df.groupBy("ID").count().filter(F.col("count") > 1).count()
print(f"\n Duplicate IDs: {duplicates_count}")

# Check data ranges
print("\n Data Statistics:")
display(raw_df.describe())

# Show rows with nulls (if any)
null_rows = raw_df.filter(
    F.col("ID").isNull() | 
    F.col("Customer_Name").isNull() | 
    F.col("City").isNull() | 
    F.col("Age").isNull()
)

if null_rows.count() > 0:
    print(f"\n Found {null_rows.count()} rows with missing values:")
    display(null_rows)
else:
    print("\n No missing values found!")

Checking data quality issues...


 Missing Values Count:


ID,Customer_Name,City,Age,Email
0,2,0,2,2



 Duplicate IDs: 0

 Data Statistics:


summary,ID,Customer_Name,City,Age,Email
count,20,18,20,18,18
mean,10.5,null,null,37.611111111111114,null
stddev,5.916079783099616,null,null,11.215214922222133,null
min,1,Ajay,Bangalore,21,ajay@gmail.com
max,20,Vikas,pune,60,vikas@gmail.com



 Found 4 rows with missing values:


ID,Customer_Name,City,Age,Email
3,null,Delhi,42,priya@gmail.com
5,Rohit,Bangalore,null,rohit@gmail.com
13,Kavya,Chennai,null,kavya@gmail.com
18,null,Chennai,48,manoj@gmail.com


In [0]:
print("Applying data cleansing transformations...\n")

# Start with raw data
cleaned_df = raw_df

# Transformation 1: Drop rows where ID is null (critical field)
print("\n rows with missing ID.")
before_count = cleaned_df.count()
cleaned_df = cleaned_df.filter(F.col("ID").isNotNull())
after_count = cleaned_df.count()
print(f"Removed {before_count - after_count} rows")

# Transformation 2: Fill missing Age with median
print("\n Filling missing Age with median...")
median_age = cleaned_df.approxQuantile("Age", [0.5], 0.01)[0]
print(f" Median age: {median_age}")
cleaned_df = cleaned_df.fillna({"Age": median_age})

# Transformation 3: Fill missing text fields with 'Unknown'
print("\n Filling missing text fields with 'Unknown'...")
fill_dict = {
    "Customer_Name": "Unknown",
    "City": "Unknown"
}

# Conditionally fill Email based on widget parameter
if fill_email == "yes":
    fill_dict["Email"] = "no-email@unknown.com"
    print("   Filling missing emails with 'no-email@unknown.com'")
else:
    print("   Keeping Email nulls as-is (parameter: fill_missing_email=no)")

cleaned_df = cleaned_df.fillna(fill_dict)

# Transformation 4: Trim whitespace from all string columns
print("\n Trimming whitespace from text fields...")
cleaned_df = cleaned_df.withColumns({
    "Customer_Name": F.trim(F.col("Customer_Name")),
    "City": F.trim(F.col("City"))
})

# Transformation 5: Standardize City names to title case
print("\n Standardizing City names to title case...")
cleaned_df = cleaned_df.withColumn("City", F.initcap(F.col("City")))

# Transformation 6: Remove duplicates based on ID (keep first occurrence)
print("\n Removing duplicate IDs...")
before_dedup = cleaned_df.count()
cleaned_df = cleaned_df.dropDuplicates(["ID"])
after_dedup = cleaned_df.count()
print(f"   Removed {before_dedup - after_dedup} duplicate records")

# Transformation 7: Validate Age is positive
print("\n Filtering out negative ages...")
before_age_filter = cleaned_df.count()
cleaned_df = cleaned_df.filter(F.col("Age") > 0)
after_age_filter = cleaned_df.count()
print(f"   Removed {before_age_filter - after_age_filter} records with invalid age")


print(f"\n Data cleansing complete!")
print(f"Final record count: {cleaned_df.count()}")
print("\n Cleaned data sample:")
display(cleaned_df.limit(10))

Applying data cleansing transformations...


 rows with missing ID.
Removed 0 rows

 Filling missing Age with median...
 Median age: 35.0

 Filling missing text fields with 'Unknown'...
   Filling missing emails with 'no-email@unknown.com'

 Trimming whitespace from text fields...

 Standardizing City names to title case...

 Removing duplicate IDs...
   Removed 0 duplicate records

 Filtering out negative ages...
   Removed 0 records with invalid age

 Data cleansing complete!
Final record count: 20

 Cleaned data sample:


ID,Customer_Name,City,Age,Email
12,Rahul,Mumbai,27,rahul@gmail.com
18,Unknown,Chennai,48,manoj@gmail.com
16,Suresh,Delhi,55,suresh@gmail.com
5,Rohit,Bangalore,35,rohit@gmail.com
10,Vikas,Bangalore,50,vikas@gmail.com
1,Raj,Delhi,21,raj@gmail.com
3,Unknown,Delhi,42,priya@gmail.com
20,Ajay,Bangalore,60,ajay@gmail.com
2,Amit,Mumbai,35,amit@gmail.com
13,Kavya,Chennai,35,kavya@gmail.com


In [0]:
# Get output table name from widget
# Note: /mnt/delta/cleaned_customers is not accessible on Serverless
# Using Unity Catalog instead (configured via widget)
table_name = output_table

print(f"Writing cleaned data to Delta table: {table_name}\n")

# Write to Delta table
cleaned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"\n Successfully written {cleaned_df.count()} records to Delta table!")
print(f"\nTable location: {table_name}")

Writing cleaned data to Delta table: csvfiles.default.cleaned_customers


 Successfully written 20 records to Delta table!

Table location: csvfiles.default.cleaned_customers


In [0]:
print("Verifying Delta table.\n")

# Read back from Delta table
verify_df = spark.table(table_name)

print(f"\n Table: {table_name}")
print(f" Total records: {verify_df.count()}")

print("\n Schema:")
verify_df.printSchema()

print("\n Final cleaned data:")
display(verify_df)

# Summary statistics
print("\n Summary Statistics:")
display(verify_df.describe())

# Quality checks on cleaned data
print("\n Data Quality Validation:")
print(f"   - No missing IDs: {verify_df.filter(F.col('ID').isNull()).count() == 0}")
print(f"   - No duplicate IDs: {verify_df.count() == verify_df.select('ID').distinct().count()}")
print(f"   - All ages positive: {verify_df.filter(F.col('Age') <= 0).count() == 0}")
print(f"   - No missing values: {verify_df.filter(F.col('Customer_Name').isNull() | F.col('City').isNull() | F.col('Age').isNull()).count() == 0}")

Verifying Delta table.


 Table: csvfiles.default.cleaned_customers
 Total records: 20

 Schema:
root
 |-- ID: integer (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Email: string (nullable = true)


 Final cleaned data:


ID,Customer_Name,City,Age,Email
12,Rahul,Mumbai,27,rahul@gmail.com
18,Unknown,Chennai,48,manoj@gmail.com
16,Suresh,Delhi,55,suresh@gmail.com
5,Rohit,Bangalore,35,rohit@gmail.com
10,Vikas,Bangalore,50,vikas@gmail.com
1,Raj,Delhi,21,raj@gmail.com
3,Unknown,Delhi,42,priya@gmail.com
20,Ajay,Bangalore,60,ajay@gmail.com
2,Amit,Mumbai,35,amit@gmail.com
13,Kavya,Chennai,35,kavya@gmail.com



 Summary Statistics:


summary,ID,Customer_Name,City,Age,Email
count,20,20,20,20,20
mean,10.5,null,null,37.35,null
stddev,5.916079783099616,null,null,10.638930496307566,null
min,1,Ajay,Bangalore,21,ajay@gmail.com
max,20,Vikas,Pune,60,vikas@gmail.com



 Data Quality Validation:
   - No missing IDs: True
   - No duplicate IDs: True
   - All ages positive: True
   - No missing values: True


## 📝 Transformation Summary

### Data Cleansing Steps Applied:

| Step | Transformation | Purpose |
|------|----------------|----------|
| 1 | Drop rows with missing ID | ID is a critical field - cannot be null |
| 2 | Fill missing Age with median | Replace nulls with statistically representative value |
| 3 | Fill missing text with defaults | Customer_Name & City: 'Unknown', Email: 'no-email@unknown.com' |
| 4 | Trim whitespace | Remove leading/trailing spaces from text fields |
| 5 | Standardize City to title case | Ensure consistent capitalization (e.g., "delhi" → "Delhi") |
| 6 | Remove duplicate IDs | Ensure each customer appears only once |
| 7 | Filter negative ages | Remove invalid age values (Age must be > 0) |

### Data Quality Improvements:

* **Before**: Raw CSV with potential nulls, duplicates, inconsistent formats
* **After**: Clean Delta table with:
  - ✅ No missing critical fields
  - ✅ No duplicate records
  - ✅ Consistent text formatting
  - ✅ Valid data ranges
  - ✅ Delta Lake ACID guarantees

### Output Location:
**Delta Table**: `csvfiles.default.cleaned_customers`

---

## Usage Example:
```python
# Read cleaned data
cleaned_customers = spark.table("csvfiles.default.cleaned_customers")

# Query by city
delhi_customers = cleaned_customers.filter(F.col("City") == "Delhi")
```